In [17]:
!pip install transformers torch langchain pydantic accelerate

In [18]:
!pip install langchain_community
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer , AutoModelForCausalLM , pipeline
import torch
from langchain_community.llms import HuggingFacePipeline

In [19]:
model_name="Qwen/Qwen2.5-3B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 ,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [20]:
pipe=pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=400 , #### tokens of output only
    temperature = 0.2
)

llm=HuggingFacePipeline(pipeline=pipe) ### the model is ready now

In [21]:
import json
import os
import uuid

EVENTS_FILE = "events.json"


def load_events():
    if not os.path.exists(EVENTS_FILE):
        return []

    with open(EVENTS_FILE, "r") as file:
        return json.load(file)


def save_events(events):
    with open(EVENTS_FILE, "w") as file:
        json.dump(events, file, indent=4)


def schedule_management(action, date, time, event_name=None, event_id=None):
    events = load_events()

    if action == "create":
        # Check for conflicts
        for event in events:
            if event["date"] == date and event["time"] == time:
                return {
                    "success": False,
                    "conflict": True,
                    "message": "There is already an event at this date and time."
                }

        new_event = {
            "id": str(uuid.uuid4()),
            "date": date,
            "time": time,
            "event_name": event_name
        }

        events.append(new_event)
        save_events(events)

        return {
            "success": True,
            "message": "Event created successfully.",
            "event": new_event
        }

    elif action == "update":
        for event in events:
            if event["id"] == event_id:
                event["date"] = date
                event["time"] = time

                if event_name:
                    event["event_name"] = event_name

                save_events(events)

                return {
                    "success": True,
                    "message": "Event updated successfully.",
                    "event": event
                }

        return {
            "success": False,
            "message": "Event not found."
        }

    elif action == "delete":
        for event in events:
            if event["id"] == event_id:
                events.remove(event)
                save_events(events)

                return {
                    "success": True,
                    "message": "Event deleted successfully."
                }

        return {
            "success": False,
            "message": "Event not found."
        }

    return {
        "success": False,
        "message": "Invalid action."
    }

In [22]:
from datetime import datetime
from zoneinfo import ZoneInfo


def location_info(location, timezone):
    try:
        current_time = datetime.now(ZoneInfo(timezone))

        return {
            "success": True,
            "location": location,
            "timezone": timezone,
            "local_time": current_time.strftime("%Y-%m-%d %H:%M:%S")
        }

    except Exception:
        return {
            "success": False,
            "message": "Invalid or unknown timezone."
        }

In [23]:
def analytics(values):
    if not values:
        return {
            "success": False,
            "message": "The values list cannot be empty."
        }

    try:
        values = [float(value) for value in values]

        return {
            "success": True,
            "average": sum(values) / len(values),
            "maximum": max(values),
            "minimum": min(values),
            "count": len(values)
        }

    except (ValueError, TypeError):
        return {
            "success": False,
            "message": "Values must be numbers."
        }

In [24]:
TOOLS = {
    "schedule_management": schedule_management,
    "location_info": location_info,
    "analytics": analytics
}

In [25]:
SYSTEM_PROMPT = """
You are an AI agent that decides which tool to use.

Available tools:

- schedule_management:
  Create, update, or delete events and detect conflicts.

- location_info:
  Return country, timezone, and current local time for a location.

- analytics:
  Calculate average, maximum, minimum, and count.
  Parameters MUST contain only:
  {"values": [numbers]}

Return ONLY valid JSON:
{
  "intent": "string",
  "tool": "tool_name",
  "parameters": {}
}

For analytics, never include "function".
Do not use keyword matching or predefined scenarios.
Do not invent missing information.
"""

In [26]:
def ask_agent(user_input):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [27]:
result = ask_agent(
    "Calculate the average of 10, 20, 30 and 40."
)

print(result)

{
  "intent": "calculate average",
  "tool": "analytics",
  "parameters": {
    "values": [10, 20, 30, 40]
  }
}


In [28]:
decision = json.loads(result)

tool_name = decision["tool"]
parameters = decision["parameters"]

tool = TOOLS[tool_name]

tool_result = tool(**parameters)

print(tool_result)

{'success': True, 'average': 25.0, 'maximum': 40.0, 'minimum': 10.0, 'count': 4}
